### WEEK 8 FINAL ETL NOTEBOOK
### Perform final ETL (Extract, Transform, Load)
### for external dataset and map to internal Muscle Tags
### Dataset: gym recommendation.xlsx
### Output: final_dataset.json

### STEP 1 - IMPORT LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import json
import re

### STEP 2 - LOAD DATASET

In [2]:
file_path = "gym recommendation.xlsx"

df = pd.read_excel(file_path)

print("Original Shape:", df.shape)
df.head()


Original Shape: (14589, 15)


,ID,Sex,Age,Height,Weight,Hypertension,Diabetes,BMI,Level,Fitness Goal,Fitness Type,Exercises,Equipment,Diet,Recommendation
0,1,Male,18,1.68,47.5,No,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells,"Vegetables: (Carrots, Sweet Potato, and Lettuc...",Follow a regular exercise schedule. Adhere to ...
1,2,Male,18,1.68,47.5,Yes,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...","Light athletic shoes, resistance bands, and li...","Vegetables: (Tomatoes, Garlic, leafy greens, b...",Follow a regular exercise schedule. Adhere to ...
2,3,Male,18,1.68,47.5,No,Yes,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, yoga, deadlifts, bench presses, and ov...","Dumbbells, barbells and Blood glucose monitor","Vegetables: (Garlic, Roma Tomatoes, Capers and...",Follow a regular exercise schedule. Adhere to ...
3,4,Male,18,1.68,47.5,Yes,Yes,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, yoga, deadlifts, bench presses, and ov...","Light athletic shoes, resistance bands, light ...","Vegetables: (Garlic, Roma Tomatoes, Capers, Gr...",Follow a regular exercise schedule. Adhere to ...
4,5,Male,18,1.68,47.5,No,No,16.83,Underweight,Weight Gain,Muscular Fitness,"Squats, deadlifts, bench presses, and overhead...",Dumbbells and barbells,"Vegetables: (Carrots, Sweet Potato, Lettuce); ...",Follow a regular exercise schedule. Adhere to ...


### STEP 3 - DATA CLEANING

### 3.1 Remove Duplicate Rows

In [3]:
df = df.drop_duplicates()

### 3.2 Handle Missing Values

In [4]:
df["Exercises"] = df["Exercises"].fillna("")
df["Fitness Goal"] = df["Fitness Goal"].fillna("General Fitness")
df["Diet"] = df["Diet"].fillna("Standard")

### 3.3 Clean Text Columns

In [5]:
text_cols = ["Exercises", "Fitness Goal", "Diet", "Recommendation"]

for col in text_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

### 3.4 Remove Extra Spaces

In [6]:
for col in text_cols:
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)

### Validate Numeric Columns

In [7]:
df = df[df["Age"] > 0]
df = df[df["Weight"] > 0]
df = df[df["Height"] > 0]

### Reset Index

In [8]:
df = df.reset_index(drop=True)

print("Cleaned Shape:", df.shape)

Cleaned Shape: (14589, 15)


### STEP 4 - MUSCLE TAG && GIF MAPPING

In [9]:
exercise_map = {

    # Chest
    "push-up": ["chest", "triceps", "shoulders"],
    "pushups": ["chest", "triceps", "shoulders"],
    "bench press": ["chest", "triceps", "shoulders"],
    "bench presses": ["chest", "triceps", "shoulders"],
    "incline bench press": ["chest", "shoulders"],
    "chest fly": ["chest"],

    # Back
    "pull-up": ["back", "biceps"],
    "pullups": ["back", "biceps"],
    "lat pulldown": ["back", "biceps"],
    "deadlift": ["back", "hamstrings", "glutes"],
    "rows": ["back", "biceps"],

    # Legs
    "squat": ["quads", "glutes"],
    "squats": ["quads", "glutes"],
    "lunges": ["quads", "glutes"],
    "leg press": ["quads"],
    "romanian deadlift": ["hamstrings", "glutes"],

    # Shoulders
    "overhead press": ["shoulders", "triceps"],
    "shoulder press": ["shoulders"],
    "lateral raise": ["shoulders"],

    # Arms
    "bicep curl": ["biceps"],
    "curl": ["biceps"],
    "tricep pushdown": ["triceps"],
    "dips": ["triceps", "chest"],

    # Core
    "plank": ["core"],
    "crunch": ["core"],
    "sit-up": ["core"],
    "leg raise": ["core"],

    # Cardio
    "running": ["cardio"],
    "cycling": ["cardio", "legs"],
    "jump rope": ["cardio"],
    "burpees": ["cardio", "full body"]
}

In [10]:
gif_map = {

    "push-up": "https://media.tenor.com/0AVbKGY_MxMAAAAC/pushup.gif",
    "bench press": "https://media.tenor.com/x1YRMx9x6sAAAAAC/bench-press.gif",
    "incline bench press": "https://media.tenor.com/4QXnYl8WQ1QAAAAC/bench-press.gif",

    "pull-up": "https://media.tenor.com/e9R7w4xD8V4AAAAC/pull-up.gif",
    "lat pulldown": "https://media.tenor.com/7LJpP1H0rV8AAAAC/lat-pulldown.gif",
    "row": "https://media.tenor.com/6XwK6x2m6x0AAAAC/seated-row.gif",

    "squat": "https://media.tenor.com/FWlL6g0P8e8AAAAC/squat.gif",
    "lunges": "https://media.tenor.com/fHkM6g0Ew2wAAAAC/lunges.gif",
    "leg press": "https://media.tenor.com/f8M2E7vK5NMAAAAC/leg-press.gif",

    "deadlift": "https://media.tenor.com/v4J4g0q0v4AAAAAC/deadlift.gif",
    "romanian deadlift": "https://media.tenor.com/v4J4g0q0v4AAAAAC/deadlift.gif",

    "overhead press": "https://media.tenor.com/Bk2Y4T4wR3UAAAAC/shoulder-press.gif",
    "lateral raise": "https://media.tenor.com/f3wWf8sM0lQAAAAC/lateral-raise.gif",

    "bicep curl": "https://media.tenor.com/lN6X1h5P7WgAAAAC/bicep-curl.gif",
    "hammer curl": "https://media.tenor.com/lN6X1h5P7WgAAAAC/bicep-curl.gif",

    "tricep pushdown": "https://media.tenor.com/4D2vM4QhM8oAAAAC/tricep-pushdown.gif",
    "dips": "https://media.tenor.com/8u8R1i5aL0MAAAAC/dips.gif",

    "plank": "https://media.tenor.com/Jm4M4V2h1fYAAAAC/plank.gif",
    "crunch": "https://media.tenor.com/q3vQn3bN4n8AAAAC/crunch.gif",

    "burpees": "https://media.tenor.com/7u6M6n4s2NMAAAAC/burpees.gif"
}

### STEP 5 - EXTRACT MUSCLE TAGS

In [11]:
def get_muscle_tags(text):
    
    text = str(text).lower()
    tags = []

    for exercise, muscles in exercise_map.items():
        if exercise in text:
            tags.extend(muscles)

    return list(set(tags))


df["Muscle Tags"] = df["Exercises"].apply(get_muscle_tags)

In [12]:
def get_gif(text):
    
    text = str(text).lower()

    for exercise, url in gif_map.items():
        if exercise in text:
            return url

    return ""

df["GIF URL"] = df["Exercises"].apply(get_gif)

### STEP 6 - OPTIONAL QUALITY CHECK

In [13]:
no_tags = df[df["Muscle Tags"].apply(len) == 0]

print("Rows with no tags:", len(no_tags))

Rows with no tags: 1688


###  STEP 7 - FINAL DATASET FORMAT

In [14]:
final_df = df[[
    "Age",
    "Weight",
    "Height",
    "BMI",
    "Fitness Goal",
    "Exercises",
    "GIF URL",
    "Muscle Tags",
    "Diet",
    "Recommendation"
]]

In [15]:
final_df.head()

,Age,Weight,Height,BMI,Fitness Goal,Exercises,GIF URL,Muscle Tags,Diet,Recommendation
0,18,47.5,1.68,16.83,weight gain,"squats, deadlifts, bench presses, and overhead...",https://media.tenor.com/x1YRMx9x6sAAAAAC/bench...,"[hamstrings, back, triceps, shoulders, quads, ...","vegetables: (carrots, sweet potato, and lettuc...",follow a regular exercise schedule. adhere to ...
1,18,47.5,1.68,16.83,weight gain,"squats, deadlifts, bench presses, and overhead...",https://media.tenor.com/x1YRMx9x6sAAAAAC/bench...,"[hamstrings, back, triceps, shoulders, quads, ...","vegetables: (tomatoes, garlic, leafy greens, b...",follow a regular exercise schedule. adhere to ...
2,18,47.5,1.68,16.83,weight gain,"squats, yoga, deadlifts, bench presses, and ov...",https://media.tenor.com/x1YRMx9x6sAAAAAC/bench...,"[hamstrings, back, triceps, shoulders, quads, ...","vegetables: (garlic, roma tomatoes, capers and...",follow a regular exercise schedule. adhere to ...
3,18,47.5,1.68,16.83,weight gain,"squats, yoga, deadlifts, bench presses, and ov...",https://media.tenor.com/x1YRMx9x6sAAAAAC/bench...,"[hamstrings, back, triceps, shoulders, quads, ...","vegetables: (garlic, roma tomatoes, capers, gr...",follow a regular exercise schedule. adhere to ...
4,18,47.5,1.68,16.83,weight gain,"squats, deadlifts, bench presses, and overhead...",https://media.tenor.com/x1YRMx9x6sAAAAAC/bench...,"[hamstrings, back, triceps, shoulders, quads, ...","vegetables: (carrots, sweet potato, lettuce); ...",follow a regular exercise schedule. adhere to ...


### STEP 8 - EXPORT FILES

In [16]:
final_df.to_json(
    "final_dataset.json",
    orient="records",
    indent=2
)

In [17]:
final_df.to_excel(
    "final_dataset_cleaned.xlsx",
    index=False
)

print("ETL Completed Successfully")

ETL Completed Successfully


### STEP 9 - ANALYTICS (Optional)

In [18]:
all_tags = []

for row in final_df["Muscle Tags"]:
    all_tags.extend(row)

tag_series = pd.Series(all_tags)

print(tag_series.value_counts())

hamstrings    7008
back          7008
triceps       7008
shoulders     7008
quads         7008
chest         7008
glutes        7008
legs          5893
cardio        5893
Name: count, dtype: int64
